# Unit 3: Drug Discovery - Single-Geometry H2 VQE Anatomy

**Companion notebook for *What Quantum Computers Are Actually For***

**Notebook class:** toy demonstration


**Code note:** This notebook is written as teaching code rather than library code. The Quokka calls, circuit construction, and measurement post-processing stay explicit on purpose so you can inspect the mechanism end to end. A production library would factor more of this behind helpers; here we keep the moving parts visible.

This notebook keeps the chemistry story honest by working at a single bond length and using a precomputed 2-qubit reduced Hamiltonian for H2.

It does **not** build molecular integrals from scratch and it does **not** produce a full potential-energy surface. What it does show, end to end, is the VQE anatomy at one geometry: a reduced Hamiltonian, a parameterised ansatz, direct Pauli-basis measurements, and a comparison against exact diagonalisation.

**What you'll see:**
1. A reduced 2-qubit H2 Hamiltonian at one bond length
2. The Hartree-Fock reference and exact-diagonalisation benchmark for that reduced model
3. A single-parameter VQE ansatz measured in the Z, XX, and YY bases
4. A parameter sweep that compares the VQE estimate with the exact benchmark

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# --- Quokka connection ---
QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and all(isinstance(v, int) for v in payload.values()):
        return payload

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = {}
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return _decode_quokka_counts(result.json())

print(f"Connected to {QUOKKA_URL}")

## 1. Reduced H2 Hamiltonian at one geometry

We work at a single bond length, $R = 0.735$ A, and use a precomputed 2-qubit reduction of the H2 Hamiltonian.

That means the notebook is a VQE anatomy demo rather than a full ab initio chemistry pipeline. The benchmark is still honest: we diagonalise the same reduced Hamiltonian exactly and compare the VQE estimate against that exact reference.

In [ ]:
def h2_hamiltonian_coeffs() -> dict:
    """Return a precomputed 2-qubit H2 Hamiltonian at R = 0.735 A."""
    return {
        'II': -0.4804,
        'Z0': +0.3435,
        'Z1': -0.4347,
        'Z0Z1': +0.5716,
        'X0X1': +0.0910,
        'Y0Y1': +0.0910,
    }


def exact_diagonalisation_energy(coeffs: dict) -> float:
    I = np.eye(2)
    X = np.array([[0, 1], [1, 0]], dtype=float)
    Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    Z = np.array([[1, 0], [0, -1]], dtype=float)
    hamiltonian = (
        coeffs['II'] * np.kron(I, I)
        + coeffs['Z0'] * np.kron(I, Z)
        + coeffs['Z1'] * np.kron(Z, I)
        + coeffs['Z0Z1'] * np.kron(Z, Z)
        + coeffs['X0X1'] * np.kron(X, X)
        + coeffs['Y0Y1'] * np.kron(Y, Y)
    )
    return float(np.min(np.linalg.eigvalsh(hamiltonian)))


def hartree_fock_reference_energy(coeffs: dict) -> float:
    return coeffs['II'] + coeffs['Z0'] - coeffs['Z1'] - coeffs['Z0Z1']


coeffs = h2_hamiltonian_coeffs()
E_exact = exact_diagonalisation_energy(coeffs)
E_hf = hartree_fock_reference_energy(coeffs)

print('Reduced H2 Hamiltonian at R = 0.735 A:')
for term, value in coeffs.items():
    print(f"  {value:+.4f} * {term}")
print()
print(f"Exact diagonalisation benchmark: {E_exact:.4f} Ha")
print(f"Hartree-Fock reference state:    {E_hf:.4f} Ha")
print(f"Recoverable correlation energy:  {E_exact - E_hf:.4f} Ha")

## 2. One-parameter VQE ansatz with direct Pauli measurements

We start from the Hartree-Fock reference state and use a one-parameter entangling circuit.

Unlike the earlier draft, the energy estimate below measures the XX and YY terms directly with basis-rotation circuits instead of inserting them analytically. That makes the measurement story truthful for this reduced Hamiltonian.

In [ ]:
def vqe_measurement_circuit(theta: float, basis: str = 'Z') -> str:
    basis_rotation_lines = []
    if basis == 'X':
        basis_rotation_lines = ['h q[0];', 'h q[1];']
    elif basis == 'Y':
        basis_rotation_lines = ['sdg q[0];', 'h q[0];', 'sdg q[1];', 'h q[1];']

    basis_rotations = '' if not basis_rotation_lines else '\n'.join(basis_rotation_lines) + '\n'

    return f"""OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

// Hartree-Fock reference state for this qubit ordering
x q[1];

// Single-parameter entangling ansatz
ry({theta:.6f}) q[0];
cx q[0], q[1];

{basis_rotations}measure q[0] -> c[0];
measure q[1] -> c[1];
"""


def expectation_from_counts(counts: dict, observable: str) -> float:
    total = sum(counts.values())
    expectation = 0.0
    for bitstring, count in counts.items():
        b0, b1 = int(bitstring[0]), int(bitstring[1])
        z0 = 1 - 2 * b0
        z1 = 1 - 2 * b1
        if observable == 'Z0':
            eigenvalue = z0
        elif observable == 'Z1':
            eigenvalue = z1
        else:
            eigenvalue = z0 * z1
        expectation += eigenvalue * count / total
    return expectation


def compute_energy(theta: float, coeffs: dict, shots: int = 1024) -> float:
    z_counts = run_qasm(vqe_measurement_circuit(theta, basis='Z'), shots=shots)
    xx_counts = run_qasm(vqe_measurement_circuit(theta, basis='X'), shots=shots)
    yy_counts = run_qasm(vqe_measurement_circuit(theta, basis='Y'), shots=shots)

    energy = coeffs['II']
    energy += coeffs['Z0'] * expectation_from_counts(z_counts, 'Z0')
    energy += coeffs['Z1'] * expectation_from_counts(z_counts, 'Z1')
    energy += coeffs['Z0Z1'] * expectation_from_counts(z_counts, 'Z0Z1')
    energy += coeffs['X0X1'] * expectation_from_counts(xx_counts, 'XX')
    energy += coeffs['Y0Y1'] * expectation_from_counts(yy_counts, 'YY')
    return float(energy)


print(vqe_measurement_circuit(np.pi / 2, basis='Z'))

## 3. Sweep the variational parameter at this one geometry

In [ ]:
thetas = np.linspace(0, np.pi, 13)
energies = []

print('Sweeping theta across one geometry...')
for theta in thetas:
    energy = compute_energy(theta, coeffs, shots=1024)
    energies.append(energy)
    print(f"  theta = {theta:.3f} -> E = {energy:.4f} Ha")

best_idx = int(np.argmin(energies))
best_theta = float(thetas[best_idx])
best_energy = float(energies[best_idx])
error_mha = abs(best_energy - E_exact) * 1000

print()
print(f"Best theta:                 {best_theta:.3f}")
print(f"VQE minimum:                {best_energy:.4f} Ha")
print(f"Exact diagonalisation:      {E_exact:.4f} Ha")
print(f"Hartree-Fock reference:     {E_hf:.4f} Ha")
print(f"Absolute error:             {error_mha:.1f} mHa")
print('This notebook stays at one bond length; a full potential-energy surface would need a new Hamiltonian at each geometry.')

In [ ]:
# Plot the one-parameter landscape at this fixed geometry
plt.figure(figsize=(10, 6))
plt.plot(thetas, energies, 'o-', color='#009688', label='VQE on Quokka')
plt.axhline(y=E_exact, color='red', linestyle='--', label=f'Exact diagonalisation = {E_exact:.4f} Ha')
plt.axhline(y=E_hf, color='orange', linestyle=':', label=f'Hartree-Fock = {E_hf:.4f} Ha')
plt.axvline(x=best_theta, color='gray', linestyle=':', alpha=0.6)
plt.xlabel('theta')
plt.ylabel('Energy (Hartree)')
plt.title('Single-geometry H2 VQE landscape')
plt.legend()
plt.tight_layout()
plt.show()

## What's next?

- [Deep-Dive 3 - The VQE Pipeline](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/06-vqe-pipeline.md): the full molecule-to-Hamiltonian story behind this reduced demo
- Replace the precomputed coefficients with a geometry-dependent classical chemistry pipeline to move toward a real potential-energy surface
- [Unit 4 - Machine Learning](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/07-machine-learning.md): another application where the quantum computer evaluates a hard quantity and a classical optimiser does the rest